# Rall 3/2 rule: full NeuroMorpho run on Google Colab

Runs `check.py` from branch `feat/rall-3-2-rule` on about 11,000 NeuroMorpho.org
reconstructions (about 2 hours of downloads). The cache lives in your Google
Drive (`MyDrive/neuro-conjectures/rall-data`), so if Colab disconnects, run all
cells again and it resumes where it stopped.

Run the cells in order (Runtime > Run all). Cell 1 asks you to allow Drive access.
At the end, `rall-outputs.zip` (check-output.json, manifest.json and the
offline re-run comparison) downloads to your computer and a copy is saved in
`MyDrive/neuro-conjectures/`.

In [ ]:
# 1. Keep the download cache in Google Drive so a disconnect does not lose it
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Get the code (branch feat/rall-3-2-rule)
%cd /content
!test -d neuro-conjectures || git clone -q -b feat/rall-3-2-rule https://github.com/IsaacZhangg/neuro-conjectures.git
%cd /content/neuro-conjectures
!git fetch -q origin feat/rall-3-2-rule && git checkout -q feat/rall-3-2-rule && git reset -q --hard origin/feat/rall-3-2-rule
!git log --oneline -1

In [ ]:
# 3. Install the pinned NumPy (needs Python 3.12 or newer); fall back to Colab's own NumPy
import sys; print(sys.version)
!pip install -q -r requirements.txt || echo "pinned numpy not installable here; using Colab's numpy"
!python -c "import numpy; print('numpy', numpy.__version__)"

In [ ]:
# 4. Point the result's data folder at Google Drive
!mkdir -p /content/drive/MyDrive/neuro-conjectures/rall-data
!rm -rf results/rall-3-2-rule/data
!ln -s /content/drive/MyDrive/neuro-conjectures/rall-data results/rall-3-2-rule/data
!ls results/rall-3-2-rule/data | head

In [ ]:
# 5. Full run: metadata (cached after the first time), then about 11,000 SWC files
%cd /content/neuro-conjectures/results/rall-3-2-rule
!python check.py 2>&1 | grep -v "^metadata page"

In [ ]:
# 6. Confirm: re-run from the cache only and compare with the recorded output
import json, shutil, subprocess
shutil.copy('check-output.json', '/content/first-run.json')
shutil.copy('manifest.json', '/content/first-manifest.json')
subprocess.run(['python', 'check.py', '--offline'], check=False)
a = json.load(open('/content/first-run.json')); b = json.load(open('check-output.json'))
a.pop('runtime_seconds', None); b.pop('runtime_seconds', None)
same = {'check_output_identical_except_runtime': a == b,
        'manifest_identical': open('/content/first-manifest.json').read() == open('manifest.json').read()}
print(same)
shutil.copy('/content/first-run.json', 'check-output.json')  # keep the online run as the record
json.dump(same, open('offline-rerun.json', 'w'), indent=2)
p = b['analyses']['primary_5um']
print('status:', a['status'], '| pooled eta', p.get('pooled_median_eta'), p.get('pooled_ci95'),
      '| groups', p.get('qualifying_groups'), p.get('group_labels'), '| flips', a.get('robustness_flips'))

In [ ]:
# 7. Save the outputs to Drive and download them
!zip -q -j /content/rall-outputs.zip check-output.json manifest.json offline-rerun.json
!cp /content/rall-outputs.zip /content/drive/MyDrive/neuro-conjectures/
from google.colab import files
files.download('/content/rall-outputs.zip')